In [ ]:
pip install langchain langchain-google-genai wikipedia duckduckgo-search beautifulsoup4 google-generativeai


  Using cached langchain_core-0.3.68-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain_core-0.3.68-py3-none-any.whl (441 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.66
    Uninstalling langchain-core-0.3.66:
      Successfully uninstalled langchain-core-0.3.66
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from langchain.agents import initialize_agent, Tool
from langchain.agents.agent_types import AgentType
from langchain.tools import DuckDuckGoSearchRun
from langchain.utilities import WikipediaAPIWrapper
from langchain.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI
from datetime import datetime
import wikipedia
from bs4 import BeautifulSoup
import requests
import re
from pathlib import Path
from dotenv import load_dotenv

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from dotenv import load_dotenv
from pathlib import Path
import os


load_dotenv()


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)


In [12]:
# Testing the LLM
llm.invoke("Hello, how are you?")

AIMessage(content='I am doing well, thank you for asking!  How are you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--a06b34b5-54a0-45c7-8502-b9fd6525a77f-0', usage_metadata={'input_tokens': 6, 'output_tokens': 17, 'total_tokens': 23, 'input_token_details': {'cache_read': 0}})

In [11]:
from langchain.tools import tool

@tool
def extract_year_from_summary(text: str) -> str:
    """Extracts the founding year from a given text."""
    import re
    match = re.search(r"\b(19|20)\d{2}\b", text)
    if match:
        return f"Founding year is {match.group(0)}"
    return "Founding year not found"


@tool
def founding_year_tool(company: str) -> str:
    """Returns the number of years since the company was founded using Wikipedia info."""
    try:
        summary = wikipedia.summary(company)
        for line in summary.split('.'):
            if "founded" in line.lower():
                year = [int(word) for word in line.split() if word.isdigit() and 1800 < int(word) < datetime.now().year]
                if year:
                    founded_year = year[0]
                    years_ago = datetime.now().year - founded_year
                    return f"{company} was founded in {founded_year}, which is {years_ago} years ago."
        return "Could not find founding year."
    
    except Exception as e:
        return f"Error: {e}"

def news_summarizer_tool_factory(llm_instance):
    @tool
    def news_summarizer_tool(query: str) -> str:
        """Searches top 3 news articles and summarizes them using Gemini."""
        try:
            search = DuckDuckGoSearchRun()
            results = search.run(query + " site:news")
            urls = [line for line in results.split("\n") if "http" in line][:3]

            summaries = []
            for url in urls:
                try:
                    res = requests.get(url, timeout=5)
                    soup = BeautifulSoup(res.text, "html.parser")
                    text = ' '.join([p.text for p in soup.find_all('p')])

                    text = text[:4000] # Gemini has a token limit, so we truncate the text
                    

                    prompt = f"Summarize this news article in 3 bullet points:\n\n{text}"
                    summary = llm_instance.invoke(prompt).content
                    summaries.append(f" {url}\n{summary}")
                except:
                    summaries.append(f" Failed to summarize: {url}")
            return "\n\n".join(summaries)
        except Exception as e:
            return f"Error during news summarization: {e}"
    return news_summarizer_tool

# Built-in Tools
wiki_tool = Tool(
    name="Wikipedia Search",
    func=WikipediaAPIWrapper().run,
    description="Fetches a short summary from Wikipedia"
)

ddg_search = Tool(
    name="DuckDuckGo Web Search",
    func=DuckDuckGoSearchRun().run,
    description="Use this to search anything on the web"
)

news_tool = news_summarizer_tool_factory(llm)

tools = [
    wiki_tool,
    ddg_search,
    founding_year_tool,
    news_tool,
    extract_year_from_summary
]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

query = """
Get me a short summary about Nvidia, calculate how old the company is, and give a 3-point summary of its latest news.
"""

response = agent.invoke({"input": query})
print("\n\nFinal Output:\n", response["output"])



> Entering new AgentExecutor chain...
Thought: I need to get information about Nvidia from Wikipedia, then calculate its age, and finally summarize recent news about it.

Action: Wikipedia Search
Action Input: Nvidia
Observation: Page: Nvidia
Summary: Nvidia Corporation ( en-VID-ee-ə) is an American multinational corporation and technology company headquartered in Santa Clara, California, and incorporated in Delaware. Founded in 1993 by Jensen Huang (president and CEO), Chris Malachowsky, and Curtis Priem, it designs and supplies graphics processing units (GPUs), application programming interfaces (APIs) for data science and high-performance computing, and system on a chip units (SoCs) for mobile computing and the automotive market. The company is also a leading supplier of artificial intelligence (AI) hardware and software. Nvidia outsources the manufacturing of the hardware it designs.
Nvidia's professional line of GPUs are used for edge-to-cloud computing and in supercomputers and

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



Observation: Error: "nida" may refer to: 
Nida (name)
Nida, Lithuania
Nida, Oklahoma
Nida, West Virginia
Nida, Świętokrzyskie Voivodeship
Nida Plateau
Nida (river)
Nida (Roman town)
Al Nida (newspaper)
Al-Nida' (Kuwait)
Nida-yi Vatan
Nida Civic Movement
NIDA (political party)
character in the computer game Final Fantasy VIII
Niddah
Typhoon Nida (disambiguation)
National Institute of Development Administration
National Institute of Dramatic Art
National Institute on Drug Abuse
National Internet Development Agency of Korea
National Information Communications Technology Development Authority, Cambodia
Thought:Thought:The founding_year_tool failed. I'll use the founding year I already extracted from the Wikipedia summary.  Then I'll get a news summary.

Action: news_summarizer_tool
Action Input: Nvidia

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: 
Thought:Thought: I have the founding year (1993) and a news summary. I need to calculate the age and then put everything together for the final answer.

Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Wikipedia Search, DuckDuckGo Web Search, founding_year_tool, news_summarizer_tool, extract_year_from_summary].
Thought:Thought: I have the founding year (1993) from the Wikipedia summary. I'll calculate the age and combine it with the news summary.

Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Wikipedia Search, DuckDuckGo Web Search, founding_year_tool, news_summarizer_tool, extract_year_from_summary].
Thought:Thought: I have the founding year (1993) from the Wikipedia summary. I'll calculate the age and combine it with the news summary.  I'll need to calculate the age manually since the founding_year_tool didn't work correctly.

Action: None
Action Input: None
Observation: None is not a valid tool, tr

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: 
Thought:Thought: I have the founding year (1993) from the Wikipedia summary. I'll calculate the age and combine it with the news summary.  I'll need to calculate the age manually since the founding_year_tool didn't work correctly.  The current year is 2024.  Nvidia's age is 2024 - 1993 = 31 years old.  I'll need to get the news summary again as it was previously incomplete.  I'll assume the news summarizer tool is working correctly now and will provide a summary.

Action: news_summarizer_tool
Action Input: Nvidia

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: 
Thought:Thought: I have the founding year (1993) from the Wikipedia summary. I'll calculate the age and combine it with the news summary.  I'll need to calculate the age manually since the founding_year_tool didn't work correctly.  The current year is 2024.  Nvidia's age is 2024 - 1993 = 31 years old.  I'll need to get the news summary again as it was previously incomplete.  I'll assume the news summarizer tool is working correctly now and will provide a summary.  Since the news_summarizer_tool is not providing output, I will leave that section blank.


Thought: I now know the final answer.

Final Answer: Nvidia is an American multinational technology company founded in 1993.  As of 2024, the company is 31 years old.  A three-point summary of its latest news is unavailable due to tool limitations.

> Finished chain.


Final Output:
 Nvidia is an American multinational technology company founded in 1993.  As of 2024, the company is 31 years old.  A three-point summary of 